# Indonesia Import Opportunity Analysis

This notebook identifies global product trends and cross-references them with Indonesia's import data to surface high-opportunity categories for importers and market researchers.

**Data Sources:**
- [Google Trends](https://trends.google.com) via `pytrends` — global search interest over the last 12 months
- [UN Comtrade](https://comtradeplus.un.org) via `comtradeapicall` — Indonesia import values by HS2 chapter (2023, free preview)

**Outputs:**
- `data/raw/` — raw API responses as CSV
- `data/processed/` — merged analysis dataset
- `output/charts/` — 4 PNG visualizations
- `output/csv/` — top opportunities table

**How to run:** Kernel → Restart & Run All

## 1. Imports & Configuration

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import comtradeapicall
from pytrends.request import TrendReq

warnings.filterwarnings('ignore')

BASE = Path('..').resolve()
DATA_RAW       = BASE / 'data' / 'raw'
DATA_PROCESSED = BASE / 'data' / 'processed'
OUTPUT_CHARTS  = BASE / 'output' / 'charts'
OUTPUT_CSV     = BASE / 'output' / 'csv'

for p in [DATA_RAW, DATA_PROCESSED, OUTPUT_CHARTS, OUTPUT_CSV]:
    p.mkdir(parents=True, exist_ok=True)

print('Paths ready.')
print(f'  Raw data  : {DATA_RAW}')
print(f'  Processed : {DATA_PROCESSED}')
print(f'  Charts    : {OUTPUT_CHARTS}')
print(f'  CSV output: {OUTPUT_CSV}')

## 2. Google Trends — Global Product Interest

In [ ]:
KEYWORD_BATCHES = [
    ['smartphone', 'laptop', 'headphones', 'smartwatch', 'tablet'],
    ['cosmetics', 'vitamin', 'supplement', 'skincare', 'perfume'],
    ['sneakers', 'clothing', 'handbag', 'sunglasses', 'sportswear'],
    ['coffee maker', 'air fryer', 'robot vacuum', 'blender', 'rice cooker'],
]

pytrends = TrendReq(hl='en-US', tz=360)
all_trends = []

for i, batch in enumerate(KEYWORD_BATCHES):
    print(f'Fetching batch {i+1}/{len(KEYWORD_BATCHES)}: {batch}')
    try:
        pytrends.build_payload(batch, timeframe='today 12-m', geo='')
        df = pytrends.interest_over_time()
        if not df.empty:
            df = df.drop(columns=['isPartial'], errors='ignore')
            means = df.mean().reset_index()
            means.columns = ['keyword', 'avg_interest']
            all_trends.append(means)
        else:
            print(f'  Warning: empty response for batch {i+1}')
    except Exception as e:
        print(f'  Error fetching batch {i+1}: {e}')
    time.sleep(2)

if all_trends:
    trends_df = pd.concat(all_trends, ignore_index=True)
    trends_df = trends_df.sort_values('avg_interest', ascending=False).reset_index(drop=True)
    trends_df.to_csv(DATA_RAW / 'google_trends_raw.csv', index=False)
    print(f'\nSaved {len(trends_df)} keywords to google_trends_raw.csv')
    display(trends_df)
else:
    print('No trend data collected. Check your internet connection.')
    trends_df = pd.DataFrame(columns=['keyword', 'avg_interest'])

## 3. UN Comtrade — Indonesia Import Data (Free Preview)

In [ ]:
HS2_CATEGORY_MAP = {
    '84': 'Machinery & Appliances',
    '85': 'Electronics',
    '27': 'Mineral Fuels & Oil',
    '72': 'Iron & Steel',
    '87': 'Vehicles',
    '39': 'Plastics',
    '90': 'Optical & Medical Instruments',
    '29': 'Chemicals',
    '10': 'Cereals & Grains',
    '23': 'Food Preparations',
    '30': 'Pharmaceuticals',
    '33': 'Cosmetics & Perfume',
    '61': 'Clothing (Knitted)',
    '62': 'Clothing (Woven)',
    '64': 'Footwear',
    '94': 'Furniture & Lighting',
    '73': 'Iron & Steel Articles',
    '76': 'Aluminium',
    '38': 'Chemical Products',
    '48': 'Paper & Paperboard',
}

print('Fetching Indonesia import data from UN Comtrade (free preview)...')
try:
    raw = comtradeapicall.previewFinalData(
        typeCode='C',
        freqCode='A',
        clCode='HS',
        period='2023',
        reporterCode='360',
        cmdCode='AG2',
        flowCode='M',
        partnerCode=None,
        partner2Code=None,
        customsCode=None,
        motCode=None,
        maxRecords=500,
        format_output='JSON',
        aggregateBy=None,
        breakdownMode='classic',
        countOnly=None,
        includeDesc=True
    )
    
    if raw is not None and not raw.empty:
        imports_df = raw[['cmdCode', 'cmdDesc', 'primaryValue', 'qty']].copy()
        imports_df.columns = ['hs2', 'hs_desc', 'trade_value_usd', 'qty']
        imports_df['hs2'] = imports_df['hs2'].astype(str).str.zfill(2)
        imports_df['category'] = imports_df['hs2'].map(HS2_CATEGORY_MAP).fillna(imports_df['hs_desc'])
        imports_df = imports_df.dropna(subset=['trade_value_usd'])
        imports_df = imports_df.sort_values('trade_value_usd', ascending=False).reset_index(drop=True)
        imports_df.to_csv(DATA_RAW / 'comtrade_indonesia_raw.csv', index=False)
        print(f'Saved {len(imports_df)} HS2 chapters to comtrade_indonesia_raw.csv')
        display(imports_df.head(10))
    else:
        print('Warning: Comtrade returned empty data. Using sample data for demonstration.')
        imports_df = pd.DataFrame({
            'hs2': ['85','84','87','27','39','90','30','33','62','64'],
            'hs_desc': ['Electrical machinery','Machinery','Vehicles','Mineral fuels','Plastics',
                        'Optical instruments','Pharmaceuticals','Cosmetics','Clothing','Footwear'],
            'trade_value_usd': [18e9, 14e9, 10e9, 8e9, 5e9, 4e9, 3e9, 2e9, 1.5e9, 1e9],
            'qty': [None]*10
        })
        imports_df['category'] = imports_df['hs2'].map(HS2_CATEGORY_MAP).fillna(imports_df['hs_desc'])

except Exception as e:
    print(f'Comtrade API error: {e}')
    print('Using sample data for demonstration.')
    imports_df = pd.DataFrame({
        'hs2': ['85','84','87','27','39','90','30','33','62','64'],
        'hs_desc': ['Electrical machinery','Machinery','Vehicles','Mineral fuels','Plastics',
                    'Optical instruments','Pharmaceuticals','Cosmetics','Clothing','Footwear'],
        'trade_value_usd': [18e9, 14e9, 10e9, 8e9, 5e9, 4e9, 3e9, 2e9, 1.5e9, 1e9],
        'qty': [None]*10
    })
    imports_df['category'] = imports_df['hs2'].map(HS2_CATEGORY_MAP).fillna(imports_df['hs_desc'])

## 4. Data Processing & Opportunity Scoring

In [ ]:
KEYWORD_TO_CATEGORY = {
    'smartphone':   'Electronics',
    'laptop':       'Electronics',
    'headphones':   'Electronics',
    'smartwatch':   'Electronics',
    'tablet':       'Electronics',
    'cosmetics':    'Cosmetics & Perfume',
    'vitamin':      'Pharmaceuticals',
    'supplement':   'Pharmaceuticals',
    'skincare':     'Cosmetics & Perfume',
    'perfume':      'Cosmetics & Perfume',
    'sneakers':     'Footwear',
    'clothing':     'Clothing (Woven)',
    'handbag':      'Clothing (Woven)',
    'sunglasses':   'Optical & Medical Instruments',
    'sportswear':   'Clothing (Knitted)',
    'coffee maker': 'Machinery & Appliances',
    'air fryer':    'Machinery & Appliances',
    'robot vacuum': 'Machinery & Appliances',
    'blender':      'Machinery & Appliances',
    'rice cooker':  'Machinery & Appliances',
}

if not trends_df.empty:
    trends_df['category'] = trends_df['keyword'].map(KEYWORD_TO_CATEGORY)
    category_trends = (
        trends_df.groupby('category')['avg_interest']
        .mean()
        .reset_index()
        .rename(columns={'avg_interest': 'trend_score'})
    )
else:
    category_trends = pd.DataFrame({
        'category': ['Electronics', 'Cosmetics & Perfume', 'Pharmaceuticals',
                     'Footwear', 'Machinery & Appliances', 'Clothing (Woven)',
                     'Optical & Medical Instruments', 'Clothing (Knitted)'],
        'trend_score': [72, 55, 48, 44, 38, 35, 30, 28]
    })

merged = category_trends.merge(imports_df[['category', 'trade_value_usd']], on='category', how='inner')

merged['opportunity_score'] = merged['trend_score'] / np.log1p(merged['trade_value_usd'] / 1e9)
merged['trade_value_bn'] = merged['trade_value_usd'] / 1e9
merged = merged.sort_values('opportunity_score', ascending=False).reset_index(drop=True)

merged.to_csv(DATA_PROCESSED / 'merged_analysis.csv', index=False)
print(f'Saved merged analysis ({len(merged)} categories) to merged_analysis.csv')
display(merged[['category', 'trend_score', 'trade_value_bn', 'opportunity_score']].round(2))

## 5. Visualizations

In [ ]:
COLORS = {
    'primary':   '#2563EB',
    'secondary': '#16A34A',
    'accent':    '#DC2626',
    'neutral':   '#6B7280',
}

def save_chart(fig, filename):
    path = OUTPUT_CHARTS / filename
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {path.name}')

In [ ]:
top_trends = trends_df.nlargest(10, 'avg_interest') if not trends_df.empty else category_trends.nlargest(10, 'trend_score').rename(columns={'trend_score': 'avg_interest', 'category': 'keyword'})

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_trends['keyword'][::-1], top_trends['avg_interest'][::-1],
               color=COLORS['primary'], alpha=0.85, edgecolor='white')
ax.set_xlabel('Average Google Trends Interest (0–100)', fontsize=11)
ax.set_title('Top 10 Trending Product Keywords — Global (Last 12 Months)', fontsize=13, fontweight='bold')
ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9)
ax.set_xlim(0, top_trends['avg_interest'].max() * 1.15)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
save_chart(fig, 'top_trends.png')

fig, ax = plt.subplots(figsize=(10, 6))
top_imports = imports_df.nlargest(10, 'trade_value_usd')
bars = ax.barh(top_imports['category'][::-1], top_imports['trade_value_usd'][::-1] / 1e9,
               color=COLORS['secondary'], alpha=0.85, edgecolor='white')
ax.set_xlabel('Import Value (USD Billion)', fontsize=11)
ax.set_title('Indonesia Top Import Categories by Value — 2023 (HS2)', fontsize=13, fontweight='bold')
ax.bar_label(bars, fmt='$%.1fB', padding=3, fontsize=9)
ax.set_xlim(0, top_imports['trade_value_usd'].max() / 1e9 * 1.2)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
save_chart(fig, 'import_values.png')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

sc = ax.scatter(
    merged['trade_value_bn'],
    merged['trend_score'],
    s=merged['opportunity_score'] * 40,
    c=merged['opportunity_score'],
    cmap='RdYlGn',
    alpha=0.8,
    edgecolors='grey',
    linewidths=0.5
)

for _, row in merged.iterrows():
    ax.annotate(
        row['category'],
        (row['trade_value_bn'], row['trend_score']),
        textcoords='offset points', xytext=(6, 4),
        fontsize=8, color='#1F2937'
    )

mid_x = merged['trade_value_bn'].median()
mid_y = merged['trend_score'].median()
ax.axvline(mid_x, color='grey', linestyle='--', alpha=0.4, linewidth=1)
ax.axhline(mid_y, color='grey', linestyle='--', alpha=0.4, linewidth=1)
ax.text(0.02, 0.97, 'OPPORTUNITY\n(high trend, low imports)', transform=ax.transAxes,
        fontsize=8, color='#16A34A', va='top', fontweight='bold')
ax.text(0.98, 0.97, 'ESTABLISHED\n(high trend, high imports)', transform=ax.transAxes,
        fontsize=8, color='#2563EB', va='top', ha='right', fontweight='bold')

plt.colorbar(sc, ax=ax, label='Opportunity Score')
ax.set_xscale('log')
ax.set_xlabel('Indonesia Import Value (USD Billion, log scale)', fontsize=11)
ax.set_ylabel('Global Trend Score (Google Trends avg)', fontsize=11)
ax.set_title('Import Opportunity Matrix: Global Demand vs Indonesia Import Value', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
save_chart(fig, 'opportunity_scatter.png')

In [ ]:
top10 = merged.nlargest(10, 'opportunity_score')

fig, ax = plt.subplots(figsize=(11, 6))
norm = plt.Normalize(top10['opportunity_score'].min(), top10['opportunity_score'].max())
cmap = plt.cm.get_cmap('YlOrRd')
bar_colors = [cmap(norm(v)) for v in top10['opportunity_score']]

bars = ax.barh(top10['category'][::-1], top10['opportunity_score'][::-1],
               color=bar_colors[::-1], edgecolor='white', linewidth=0.5)
ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
ax.set_xlabel('Opportunity Score  (trend ÷ log(import value))', fontsize=11)
ax.set_title('Top 10 Indonesia Import Opportunities\n(High Global Demand × Relatively Low Current Imports)', 
             fontsize=13, fontweight='bold')
ax.set_xlim(0, top10['opportunity_score'].max() * 1.2)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
save_chart(fig, 'top_opportunities.png')

## 6. Results Table

In [ ]:
results = merged[['category', 'trend_score', 'trade_value_bn', 'opportunity_score']].copy()
results.columns = ['Category', 'Global Trend Score', 'Indonesia Import (USD B)', 'Opportunity Score']
results = results.round(2).reset_index(drop=True)
results.index += 1

results.to_csv(OUTPUT_CSV / 'top_opportunities.csv')
print(f'Saved to {OUTPUT_CSV / "top_opportunities.csv"}')

display(
    results.style
    .background_gradient(subset=['Opportunity Score'], cmap='YlOrRd')
    .background_gradient(subset=['Global Trend Score'], cmap='Blues')
    .format({'Indonesia Import (USD B)': '${:.2f}B', 'Opportunity Score': '{:.2f}', 'Global Trend Score': '{:.1f}'})
)

## 7. Summary

In [ ]:
top3 = merged.nlargest(3, 'opportunity_score')
print('=' * 60)
print('INDONESIA IMPORT OPPORTUNITY SUMMARY')
print('=' * 60)
print(f'Analysis period : Google Trends (last 12 months) + Comtrade 2023')
print(f'Keywords tracked: {len(trends_df)} product keywords')
print(f'HS2 categories  : {len(imports_df)} import chapters')
print(f'Matched pairs   : {len(merged)} categories')
print()
print('Top 3 Import Opportunities:')
for i, (_, row) in enumerate(top3.iterrows(), 1):
    print(f'  {i}. {row["category"]}')
    print(f'     Trend score: {row["trend_score"]:.1f}  |  '
          f'Import value: ${row["trade_value_bn"]:.2f}B  |  '
          f'Opportunity: {row["opportunity_score"]:.2f}')
print()
print('Charts saved to : output/charts/')
print('Data saved to   : output/csv/top_opportunities.csv')